<a href="https://colab.research.google.com/github/vkchennuru/raw_scaffolding-code-evaluation/blob/main/notebooks/Notebook02_Raw_AI_Code_Collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 02 – Raw AI Code Collection

## Project

**From Generation to Justification: Transformation in Introductory Programming Education**

---

### Purpose

This notebook organizes the collection of raw AI-generated Python programs for the benchmark dataset prepared in Notebook 01.

Each benchmark problem will receive three independent AI-generated solutions generated using identical prompts.

The collected programs will be stored for subsequent evaluation in functional correctness testing, Halstead metrics, AST analysis, and statistical analysis.

---

### Inputs

- dataset/problems.csv
- prompts/raw/*.md

### Outputs

- generated_code/raw/
- results/raw_generation_summary.csv

---

Author: **Dr. C. V. Krishnaveni**

Institution:
SKR & SKR Government College for Women (Autonomous), Kadapa

Version: 1.0

In [30]:
# ============================================================
# Cell 2 : Imports
# ============================================================

import os
from pathlib import Path
import pandas as pd
from datetime import datetime

print("="*60)
print("Notebook 02")
print("="*60)

print("Execution Time :", datetime.now())
print("Pandas Version :", pd.__version__)

Notebook 02
Execution Time : 2026-07-23 05:19:56.833639
Pandas Version : 2.2.2


In [31]:
# Clone the repository (only if it doesn't already exist)

from pathlib import Path
import os

REPO_NAME = "raw_scaffolding-code-evaluation"

if not Path(f"/content/{REPO_NAME}").exists():
    !git clone https://github.com/vkchennuru/raw_scaffolding-code-evaluation.git

%cd /content/raw_scaffolding-code-evaluation

print("Current Working Directory:")
!pwd

/content/raw_scaffolding-code-evaluation
Current Working Directory:
/content/raw_scaffolding-code-evaluation


In [32]:
# ============================================================
# Cell 3 : Repository Configuration
# ============================================================

from pathlib import Path

PROJECT_ROOT = Path("/content/raw_scaffolding-code-evaluation")

DATASET_DIR = PROJECT_ROOT / "dataset"

PROMPTS_DIR = PROJECT_ROOT / "prompts"

RAW_PROMPTS_DIR = PROMPTS_DIR / "raw"

GENERATED_CODE_DIR = PROJECT_ROOT / "generated_code"

RAW_CODE_DIR = GENERATED_CODE_DIR / "raw"

RESULTS_DIR = PROJECT_ROOT / "results"

print("Project Root:", PROJECT_ROOT)

Project Root: /content/raw_scaffolding-code-evaluation


In [33]:
# ============================================================
# Cell 4 : Verify Repository
# ============================================================

folders = [
    DATASET_DIR,
    RAW_PROMPTS_DIR,
    RAW_CODE_DIR,
    RESULTS_DIR
]

print("="*60)
print("Repository Verification")
print("="*60)

for folder in folders:

    if folder.exists():

        print("✓", folder.name)

    else:

        print("✗", folder.name)

Repository Verification
✓ dataset
✓ raw
✓ raw
✓ results


In [34]:
!find /content/raw_scaffolding-code-evaluation -maxdepth 3 -type f

/content/raw_scaffolding-code-evaluation/results/.gitkeep
/content/raw_scaffolding-code-evaluation/results/raw_generation_summary.csv
/content/raw_scaffolding-code-evaluation/figures/.gitkeep
/content/raw_scaffolding-code-evaluation/generated_code/raw/.gitkeep
/content/raw_scaffolding-code-evaluation/generated_code/scaffolded/.gitkeep
/content/raw_scaffolding-code-evaluation/dataset/problems.csv
/content/raw_scaffolding-code-evaluation/dataset/.gitkeep
/content/raw_scaffolding-code-evaluation/LICENSE
/content/raw_scaffolding-code-evaluation/.git/description
/content/raw_scaffolding-code-evaluation/.git/packed-refs
/content/raw_scaffolding-code-evaluation/.git/HEAD
/content/raw_scaffolding-code-evaluation/.git/hooks/push-to-checkout.sample
/content/raw_scaffolding-code-evaluation/.git/hooks/pre-commit.sample
/content/raw_scaffolding-code-evaluation/.git/hooks/commit-msg.sample
/content/raw_scaffolding-code-evaluation/.git/hooks/pre-applypatch.sample
/content/raw_scaffolding-code-evaluat

In [35]:
# ============================================================
# Cell 5 : Create or Load Benchmark Dataset
# ============================================================

import pandas as pd

dataset_file = DATASET_DIR / "problems.csv"

if dataset_file.exists():
    dataset = pd.read_csv(dataset_file)
    print("✓ Existing dataset loaded.")

else:
    print("Dataset not found.")
    print("Creating benchmark dataset...")

    problems = [

        {
            "problem_id":"P001",
            "title":"Sum of Two Integers",
            "category":"Arithmetic",
            "difficulty":"Easy",
            "problem_statement":"Read two integers and print their sum."
        },

        {
            "problem_id":"P002",
            "title":"Largest of Three Numbers",
            "category":"Decision Making",
            "difficulty":"Easy",
            "problem_statement":"Read three numbers and print the largest."
        },

        {
            "problem_id":"P003",
            "title":"Even or Odd",
            "category":"Decision Making",
            "difficulty":"Easy",
            "problem_statement":"Determine whether a given integer is even or odd."
        }

        # We will continue adding the remaining problems.
    ]

    dataset = pd.DataFrame(problems)

    dataset.to_csv(dataset_file, index=False)

    print("✓ Dataset created successfully.")

print()
print(dataset.shape)
display(dataset)

✓ Existing dataset loaded.

(8, 5)


,problem_id,title,category,difficulty,problem_statement
0,P001,Sum of Two Integers,Arithmetic,Easy,Read two integers and print their sum.
1,P002,Largest of Three Numbers,Decision Making,Easy,Read three numbers and print the largest.
2,P003,Even or Odd,Decision Making,Easy,Determine whether a given integer is even or odd.
3,P004,Factorial of a Number,Loops,Easy,Read an integer and print its factorial.
4,P005,Prime Number Check,Loops,Easy,Determine whether the given integer is prime.
5,P006,Fibonacci Series,Loops,Easy,Print the first N Fibonacci numbers.
6,P007,Palindrome Number,Loops,Easy,Determine whether the given integer is a palin...
7,P008,Reverse a Number,Loops,Easy,Reverse the digits of a given integer.


In [36]:
# ============================================================
# Cell 6 : Generate Standardized Prompt Files
# ============================================================

RAW_PROMPTS_DIR.mkdir(parents=True, exist_ok=True)

for _, row in dataset.iterrows():

    prompt = f"""
You are an expert Python programmer.

Solve the following programming problem.

Problem ID:
{row['problem_id']}

Title:
{row['title']}

Problem Statement:
{row['problem_statement']}

Instructions:

1. Write a complete Python program.
2. Read input from standard input().
3. Print output using print().
4. Do NOT include explanations.
5. Do NOT include markdown.
6. Return ONLY Python code.
"""

    filename = RAW_PROMPTS_DIR / f"{row['problem_id']}.md"

    with open(filename, "w", encoding="utf-8") as f:
        f.write(prompt)

print(f"✓ Generated {len(dataset)} prompt files.")

✓ Generated 8 prompt files.


In [37]:
# ============================================================
# Cell 7 : Verify Prompt Files
# ============================================================

prompt_files = sorted(RAW_PROMPTS_DIR.glob("*.md"))

print("Prompt Files")

for file in prompt_files:
    print(file.name)

print("\nTotal Prompt Files:", len(prompt_files))

Prompt Files
P001.md
P002.md
P003.md
P004.md
P005.md
P006.md
P007.md
P008.md

Total Prompt Files: 8


In [38]:
# ============================================================
# Cell 8 : Create Solution Folders
# ============================================================

for problem_id in dataset["problem_id"]:

    folder = RAW_CODE_DIR / problem_id

    folder.mkdir(parents=True, exist_ok=True)

print("✓ Folder structure created.")

✓ Folder structure created.


In [39]:
# ============================================================
# Cell 9 : Verify Folder Structure
# ============================================================

print("Generated Code Folder Structure\n")

for folder in sorted(RAW_CODE_DIR.iterdir()):

    if folder.is_dir():
        print(folder.name)

Generated Code Folder Structure

P001
P002
P003
P004
P005
P006
P007
P008


In [40]:
# ============================================================
# Cell 10 : Create Code Collection Tracker
# ============================================================

tracker = dataset[["problem_id", "title"]].copy()

tracker["solution_1"] = False
tracker["solution_2"] = False
tracker["solution_3"] = False

tracker.to_csv(RESULTS_DIR / "raw_generation_summary.csv", index=False)

print("✓ Tracking file created.")

display(tracker)

✓ Tracking file created.


,problem_id,title,solution_1,solution_2,solution_3
0,P001,Sum of Two Integers,False,False,False
1,P002,Largest of Three Numbers,False,False,False
2,P003,Even or Odd,False,False,False
3,P004,Factorial of a Number,False,False,False
4,P005,Prime Number Check,False,False,False
5,P006,Fibonacci Series,False,False,False
6,P007,Palindrome Number,False,False,False
7,P008,Reverse a Number,False,False,False


In [41]:
# ============================================================
# Cell 11 : Expand Benchmark Dataset
# ============================================================

additional_problems = [

    {
        "problem_id":"P004",
        "title":"Factorial of a Number",
        "category":"Loops",
        "difficulty":"Easy",
        "problem_statement":"Read an integer and print its factorial."
    },

    {
        "problem_id":"P005",
        "title":"Prime Number Check",
        "category":"Loops",
        "difficulty":"Easy",
        "problem_statement":"Determine whether the given integer is prime."
    },

    {
        "problem_id":"P006",
        "title":"Fibonacci Series",
        "category":"Loops",
        "difficulty":"Easy",
        "problem_statement":"Print the first N Fibonacci numbers."
    },

    {
        "problem_id":"P007",
        "title":"Palindrome Number",
        "category":"Loops",
        "difficulty":"Easy",
        "problem_statement":"Determine whether the given integer is a palindrome."
    },

    {
        "problem_id":"P008",
        "title":"Reverse a Number",
        "category":"Loops",
        "difficulty":"Easy",
        "problem_statement":"Reverse the digits of a given integer."
    }

]

new_df = pd.DataFrame(additional_problems)

dataset = pd.concat([dataset, new_df], ignore_index=True)

dataset.to_csv(DATASET_DIR / "problems.csv", index=False)

print("Dataset Updated")

print(dataset.shape)

display(dataset.tail())

Dataset Updated
(13, 5)


,problem_id,title,category,difficulty,problem_statement
8,P004,Factorial of a Number,Loops,Easy,Read an integer and print its factorial.
9,P005,Prime Number Check,Loops,Easy,Determine whether the given integer is prime.
10,P006,Fibonacci Series,Loops,Easy,Print the first N Fibonacci numbers.
11,P007,Palindrome Number,Loops,Easy,Determine whether the given integer is a palin...
12,P008,Reverse a Number,Loops,Easy,Reverse the digits of a given integer.


In [42]:
# ============================================================
# Cell 12 : Regenerate Prompt Files
# ============================================================

RAW_PROMPTS_DIR.mkdir(parents=True, exist_ok=True)

count = 0

for _, row in dataset.iterrows():

    prompt = f"""
You are an expert Python programmer.

Solve the following programming problem.

Problem ID:
{row['problem_id']}

Title:
{row['title']}

Problem Statement:
{row['problem_statement']}

Instructions:

1. Write a complete Python program.
2. Read input using input().
3. Print output using print().
4. Return ONLY Python code.
5. Do NOT include explanations.
6. Do NOT include markdown.
"""

    filename = RAW_PROMPTS_DIR / f"{row['problem_id']}.md"

    with open(filename, "w", encoding="utf-8") as f:
        f.write(prompt)

    count += 1

print(f"✓ {count} prompt files generated.")

✓ 13 prompt files generated.


In [43]:
# ============================================================
# Cell 13 : Create Solution Folders
# ============================================================

for problem_id in dataset["problem_id"]:

    folder = RAW_CODE_DIR / problem_id

    folder.mkdir(parents=True, exist_ok=True)

print("✓ Solution folders created.")

✓ Solution folders created.


In [44]:
# ============================================================
# Cell 14 : Update Tracking File
# ============================================================

tracker = dataset[["problem_id", "title"]].copy()

tracker["solution_1"] = False
tracker["solution_2"] = False
tracker["solution_3"] = False

tracker.to_csv(
    RESULTS_DIR / "raw_generation_summary.csv",
    index=False
)

display(tracker)

print("✓ Tracking file updated.")

,problem_id,title,solution_1,solution_2,solution_3
0,P001,Sum of Two Integers,False,False,False
1,P002,Largest of Three Numbers,False,False,False
2,P003,Even or Odd,False,False,False
3,P004,Factorial of a Number,False,False,False
4,P005,Prime Number Check,False,False,False
5,P006,Fibonacci Series,False,False,False
6,P007,Palindrome Number,False,False,False
7,P008,Reverse a Number,False,False,False
8,P004,Factorial of a Number,False,False,False
9,P005,Prime Number Check,False,False,False


✓ Tracking file updated.


In [45]:
# ============================================================
# Cell 15 : Notebook Verification
# ============================================================

print("=" * 60)

print("Dataset File:")
print((DATASET_DIR / "problems.csv").exists())

print("\nPrompt Files:")
print(len(list(RAW_PROMPTS_DIR.glob("*.md"))))

print("\nProblem Folders:")
print(len([x for x in RAW_CODE_DIR.iterdir() if x.is_dir()]))

print("\nTracking File:")
print((RESULTS_DIR / "raw_generation_summary.csv").exists())

print("=" * 60)

Dataset File:
True

Prompt Files:
8

Problem Folders:
8

Tracking File:
True
